In [1]:
from pyspark.sql import SparkSession

#### Basic SparkSession / SparkContext creation – the foundation for every demo.


In [2]:
# SparkSession is the unified entry point (includes SparkContext)
spark = (
    SparkSession.builder.appName("SparkDemo")
    .master("local[*]")
    .config("spark.executor.instances", "2")
    .config("spark.executor.memory", "2g")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

25/11/10 10:12:52 WARN Utils: Your hostname, Tonys-MacBook-Air-2.local resolves to a loopback address: 127.0.0.1; using 192.168.59.25 instead (on interface en0)
25/11/10 10:12:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/10 10:12:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext
print(f"{f'Spark version':<20}", spark.version)
print(f"{f'App name':<20}", sc.appName)
print(f"{f'Master':<20}", sc.master)

Spark version        3.5.7
App name             SparkDemo
Master               local[*]


In [7]:
# Small sanity check
data = sc.parallelize([1, 2, 3, 4, 5])
print("Sum via RDD :", data.reduce(lambda a, b: a + b))

Sum via RDD : 15


In [8]:
spark.stop()

#### Spark accumulator for counting malformed records.

In [2]:
spark = (
    SparkSession.builder.appName("SparkDemo")
    .master("local[*]")
    .config("spark.executor.instances", "2")
    .config("spark.executor.memory", "2g")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

sc = spark.sparkContext
bad_counter = sc.accumulator(0)

25/11/10 10:55:11 WARN Utils: Your hostname, Tonys-MacBook-Air-2.local resolves to a loopback address: 127.0.0.1; using 192.168.59.25 instead (on interface en0)
25/11/10 10:55:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/10 10:55:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
def is_valid(line: str) -> bool:
    """Very simple validation – line must contain at least two comma‑separated fields."""
    return len(line.split(" ")) > 2


def parse(line: str):
    if not is_valid(line):
        bad_counter.add(1)
    return line

In [4]:
# Example log file – replace with your own path if needed
lines = sc.textFile("./data/logs_week1.txt")


_ = lines.map(parse).count()  # force evaluation
print(f"Number of malformed lines: {bad_counter.value}")

Number of malformed lines: 2


#### Broadcast a stop‑word list to all executors and filter a text file.


In [5]:
import nltk

In [6]:
nltk.download("stopwords", download_dir="./data")

[nltk_data] Downloading package stopwords to ./data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
def load_stopwords(path: str) -> set:
    with open(path, "r") as f:
        return {word.strip() for word in f.readlines()}


stop_words = sc.broadcast(set(load_stopwords("./data/corpora/stopwords/english")))

lines = sc.textFile("./data/shakespeare.txt")
words = lines.flatMap(lambda l: l.split())

In [8]:
words.take(10)

['From',
 'fairest',
 'creatures',
 'we',
 'desire',
 'increase,',
 'That',
 'thereby',
 "beauty's",
 'rose']

In [9]:
filtered = words.filter(lambda w: w.lower() not in stop_words.value)

print("Sample of filtered words:", filtered.take(10))

Sample of filtered words: ['fairest', 'creatures', 'desire', 'increase,', 'thereby', "beauty's", 'rose', 'might', 'never', 'die,']


#### Spark transformations



##### map, flatMap, filter, sample – the core RDD transformations.

In [10]:
# map → length of each line
line_lengths = lines.map(lambda l: len(l))
print("First 5 line lengths:", line_lengths.take(5))

First 5 line lengths: [44, 45, 42, 40, 47]


In [11]:
# flatMap → individual tokens
tokens = lines.flatMap(lambda l: l.split())
print("First 10 tokens:", tokens.take(10))

First 10 tokens: ['From', 'fairest', 'creatures', 'we', 'desire', 'increase,', 'That', 'thereby', "beauty's", 'rose']


In [ ]:
# filter → keep words longer than 5 chars
long_words = tokens.filter(lambda w: len(w) > 5)
print("Sample long words:", long_words.take(10))

Sample long words: ['fairest', 'creatures', 'desire', 'increase,', 'thereby', "beauty's", 'should', 'decease,', 'tender', 'memory:']


In [13]:
# sample → 1% of the long words (without replacement)
sample = long_words.sample(False, 0.01, seed=42)
print("Sample size (approx 1%):", sample.count())

Sample size (approx 1%): 2394


In [16]:
rdd = sc.parallelize([3, 1, 4, 1, 5, 9, 2, 6, 5])

# reduce – sum of all elements
total = rdd.reduce(lambda a, b: a + b)
print("reduce (sum):", total)

# fold – same as reduce but with an explicit zero value
folded = rdd.fold(0, lambda a, b: a + b)
print("fold (sum with zero):", folded)

reduce (sum): 36
fold (sum with zero): 36


##### How `aggregate` works

Suppose we have an RDD with some data: [1, 2, 3, 4]

**Instructions**

1. Initialization: Start with a `zero value` which represents the initial state of the aggregation (a tuple representing the two accumulators).
2. `SeqOp (Sequential Operation)`: Apply the seqOp function to each element in an RDD partition, reducing that partition into a single aggregated result for that partition.
3. `CombOp (Combinatory Operation)`: Combine results from different partitions using the combOp function.

**How It Works**

1. The RDD is divided into partitions (for example: `[1, 2]` and `[3, 4]`).
2. seqOp will be applied to each partition:
   1. In the first partition `[1, 2]`, it adds them up: 0 + 1 = 1 and then 1 + 2 = 3.
   2. In the second partition `[3, 4]`, it adds them up: 0 + 3 = 3 and then 3 + 4 = 7.
3. Now we have two partial results (from each partition): 3 and 7.
4. The combOp will combine these partial results: 3 + 7 = 10.

In [17]:
import numpy as np

# Create an RDD with some data
data = sc.parallelize([np.random.randint(0, 50) for _ in range(10)])
print(data.collect())

[18, 36, 28, 45, 34, 29, 0, 27, 31, 7]


In [19]:
# Define zero value: the initial state of the aggregation is a tuple (sum, count)
zeroValue = (0, 0)


# Define seqOp: function to add elements in each partition
def seqOp(acc, value):
    sum_val, count_val = acc
    return (sum_val + value, count_val + 1)


# Define combOp: function to combine results from different partitions
def combOp(acc1, acc2):
    sum1, count1 = acc1
    sum2, count2 = acc2
    return (sum1 + sum2, count1 + count2)


# Perform aggregation using `aggregate` method
result = data.aggregate(zeroValue, seqOp, combOp)
print("The result of the aggregation is:", result)

The result of the aggregation is: (255, 10)


##### Key‑based operations: mapValues, reduceByKey, groupByKey, and various joins.

In [20]:
from rich import print as rprint

In [ ]:
# Sample RDD of (user, amount) pairs
purchases = sc.parallelize([("u1", 100), ("u2", 150), ("u1", 200), ("u3", 50)])

In [22]:
# mapValues – add a tax of 10%
with_tax = purchases.mapValues(lambda amt: round(amt * 1.1, 2))
rprint("With tax:", with_tax.collect())

With tax:
[('u1', 110.0), ('u2', 165.0), ('u1', 220.0), ('u3', 55.0)]

In [24]:
# reduceByKey – total spend per user
total_by_user = purchases.reduceByKey(lambda a, b: a + b)
rprint("Total per user:", total_by_user.collect())

Total per user:
[('u2', 150), ('u1', 300), ('u3', 50)]

In [25]:
# groupByKey – list of all amounts per user (expensive!)
grouped = purchases.groupByKey().mapValues(list)
rprint("Grouped:", grouped.collect())

Grouped:
[('u2', [150]), ('u1', [100, 200]), ('u3', [50])]

In [26]:
# Joins
users = sc.parallelize([("u1", "Alice"), ("u2", "Bob"), ("u4", "Eve")])
inner = users.join(purchases)  # only matching keys
left = users.leftOuterJoin(purchases)  # keep all users
cog = users.cogroup(purchases)  # (user, (list_profiles, list_purchases))

In [27]:
rprint("Inner join:", inner.collect())
rprint("Left outer join:", left.collect())
rprint(f"Cogroup: {[(k, (list(v[0]), list(v[1]))) for k, v in cog.collect()]}")

Inner join:
[('u2', ('Bob', 150)), ('u1', ('Alice', 100)), ('u1', ('Alice', 200))]

Left outer join:
[('u4', ('Eve', None)), ('u2', ('Bob', 150)), ('u1', ('Alice', 100)), ('u1', ('Alice', 200))]

Cogroup: [('u4', (['Eve'], [])), ('u2', (['Bob'], [150])), ('u1', (['Alice'], [100, 200])), ('u3', ([], [50]))]

##### Set operations: Union, intersection and subtraction on two log files.

In [28]:
week1 = sc.textFile("./data/logs_week1.txt")  # 30 records
week2 = sc.textFile("./data/logs_week2.txt")  # 10 records

all_logs = week1.union(week2)
common_ips = week1.intersection(week2)
cleaned = all_logs.subtract(week2)

print(f"{f'All logs count:':<15s}", all_logs.count())
print(f"{f'Common messages count:':<15s}", common_ips.count())
print(f"{f'Cleaned logs count:':<15s}", cleaned.count())

All logs count: 40
Common messages count: 10
Cleaned logs count: 20


#### Spark Actions: collect, take, first, reduce, fold, aggregate.

In [30]:
rdd = sc.parallelize([3, 1, 4, 1, 5, 9, 2, 6, 5])

# collect – bring whole dataset to driver (dangerous for big data!)
all_data = rdd.collect()
print("collect:", all_data)

# take – first N elements
first_three = rdd.take(3)
print("take(3):", first_three)

# first – shortcut for take(1)
first_one = rdd.take(1)
print("first():", first_one)

collect: [3, 1, 4, 1, 5, 9, 2, 6, 5]
take(3): [3, 1, 4]
first(): [3]
